# scOPE end-to-end example

This notebook walks through the complete two-phase scOPE workflow:
1. **Phase 1** — learn latent space from bulk RNA-seq and train mutation classifiers.
2. **Phase 2** — project scRNA-seq into the bulk latent space, infer per-cell mutation probabilities.

**Data notes (BeatAML ID scheme):**
- `labId` / `seq_id`: internal `13-XXXXX` barcodes — RNA and DNA get *different* seq_ids for the same patient, so these cannot be directly joined.
- `dbgap_rnaseq_sample` (`BA*R`): RNA-seq sample IDs used in the harmonized dbGaP expression file.
- `dbgap_dnaseq_sample` (`BA*D`): WES sample IDs used in the dbGaP mutation calls file.
- The clinical file maps `BA*D` → `BA*R` and is required to join mutations to expression.

## 1. Imports & paths

In [47]:
import os

import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

from scope import BulkPipeline, SingleCellPipeline
from scope.visualization import (
    compute_umap,
    plot_mutation_probabilities,
    plot_scree,
    plot_mutation_heatmap,
)


In [48]:
# ── Paths ──────────────────────────────────────────────────────────────────
BEATAML_DIR  = "/Users/ashforda/Desktop/Pathways + Omics/scOPE/scOPE_project_overhaul/scOPE/data/BeatAML"
VANGALEN_DIR = "/Users/ashforda/Desktop/Pathways + Omics/scOPE/scOPE_project_overhaul/scOPE/data/vanGalen_AML_scRNA"
RAW_DIR      = os.path.join(BEATAML_DIR, "publicly_available_and_raw_counts")

# Harmonized dbGaP release — RNA IDs: BA*R, WES IDs: BA*D
bulk_txt   = os.path.join(RAW_DIR, "beataml_waves1to4_norm_exp_dbgap.txt")
mut_path   = os.path.join(RAW_DIR, "beataml_wes_wv1to4_mutations_dbgap.txt")
clin_path  = os.path.join(RAW_DIR, "beataml_wv1to4_clinical.xlsx")
sc_path    = os.path.join(VANGALEN_DIR, "vanGalen_anndata.h5ad")


## 2. Load & prepare bulk RNA-seq

In [49]:
# ── Peek at orientation before loading fully ───────────────────────────────
peek = pd.read_csv(bulk_txt, sep="\t", index_col=0, nrows=3)
print("Raw peek shape (nrows=3):", peek.shape)
print("Row index sample  :", peek.index[:3].tolist())
print("Column name sample:", peek.columns[:3].tolist())


Raw peek shape (nrows=3): (3, 710)
Row index sample  : ['ENSG00000000003', 'ENSG00000000419', 'ENSG00000000457']
Column name sample: ['display_label', 'description', 'biotype']


In [50]:
df_bulk = pd.read_csv(bulk_txt, sep="\t", index_col=0)

# Drop gene metadata columns before transposing
meta_cols = [c for c in df_bulk.columns if not (c.startswith('BA') and c.endswith('R'))]
if meta_cols:
    print(f"Dropping non-sample metadata columns: {meta_cols}")
    df_bulk = df_bulk.drop(columns=meta_cols)

# File is genes × samples — transpose to samples × genes
df_bulk = df_bulk.T

adata_bulk = ad.AnnData(
    X   = df_bulk.values.astype(np.float32),
    obs = pd.DataFrame(index=df_bulk.index),
    var = pd.DataFrame(index=df_bulk.columns),
)

print(f"\nBulk loaded : {adata_bulk.n_obs} samples x {adata_bulk.n_vars} genes")
print(f"Sample IDs  : {adata_bulk.obs_names[:5].tolist()}")
print(f"Gene IDs    : {adata_bulk.var_names[:5].tolist()}")


Dropping non-sample metadata columns: ['display_label', 'description', 'biotype']

Bulk loaded : 707 samples x 22843 genes
Sample IDs  : ['BA2392R', 'BA2611R', 'BA2506R', 'BA2430R', 'BA2448R']
Gene IDs    : ['ENSG00000000003', 'ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938']


In [51]:
# ── Remap Ensembl IDs → gene symbols (if needed) ──────────────────────────
if adata_bulk.var_names[0].startswith('ENSG'):
    import mygene
    mg = mygene.MyGeneInfo()
    result = mg.querymany(
        adata_bulk.var_names.tolist(),
        scopes='ensembl.gene', fields='symbol',
        species='human', as_dataframe=True
    )
    id2sym = result['symbol'].dropna().to_dict()
    adata_bulk.var_names = pd.Index([id2sym.get(g, g) for g in adata_bulk.var_names])
    adata_bulk = adata_bulk[:, ~adata_bulk.var_names.str.startswith('ENSG')].copy()
    adata_bulk.var_names_make_unique()
    print(f"After gene remapping : {adata_bulk.n_obs} samples x {adata_bulk.n_vars} genes")
else:
    print(f"Gene IDs already symbols — skipping remap. ({adata_bulk.var_names[0]!r})")
    

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
3 input query terms found dup hits:	[('ENSG00000175711', 2), ('ENSG00000215156', 2), ('ENSG00000259182', 2)]
1137 input query terms found no hit:	['ENSG00000005955', 'ENSG00000006074', 'ENSG00000006075', 'ENSG00000006114', 'ENSG00000017373', 'ENS


After gene remapping : 707 samples x 19975 genes


/opt/homebrew/Cellar/micromamba/2.5.0_1/envs/scope-dev/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


## 3. Build mutation label matrix

The dbGaP mutations file uses `dbgap_sample_id` (`BA*D`, DNA).  
The bulk expression file uses `BA*R` (RNA) IDs.  
The clinical file maps `BA*D` → `BA*R` so we can join them.

In [52]:
# ── Build BA*D → BA*R map from clinical file ───────────────────────────────
clin = pd.read_excel(clin_path)
print("Clinical columns:", clin.columns.tolist())

wes_to_rna = (
    clin[['dbgap_dnaseq_sample', 'dbgap_rnaseq_sample']]
    .dropna()
    .set_index('dbgap_dnaseq_sample')['dbgap_rnaseq_sample']
    .to_dict()
)

print(f"BA*D -> BA*R mappings available: {len(wes_to_rna)}")
print("Sample entry:", next(iter(wes_to_rna.items())))

# ── Load sample mapping to identify confirmed healthy controls ─────────────
mapping_path = os.path.join(RAW_DIR, "beataml_waves1to4_sample_mapping.xlsx")
#mapping_path = os.path.join(RAW_DIR, "beataml_wv1to4_clinical.xlsx")
mapping = pd.read_excel(mapping_path)
print("\nMapping columns:", mapping.columns.tolist())

NORMAL_CONTROL_TYPES = {'Healthy Individual BM MNC', 'Healthy pooled CD34+', 'Healthy Individual CD34+'}

normal_rna_ids = set(
    mapping.loc[mapping['rna_control'].isin(NORMAL_CONTROL_TYPES), 'dbgap_rnaseq_sample']
    .dropna()
    .unique()
)
print(f"Confirmed normal RNA sample IDs: {len(normal_rna_ids)}")


Clinical columns: ['dbgap_subject_id', 'dbgap_dnaseq_sample', 'dbgap_rnaseq_sample', 'cohort', 'used_manuscript_analyses', 'manuscript_dnaseq', 'manuscript_rnaseq', 'manuscript_inhibitor', 'consensus_sex', 'inferred_sex', 'reportedRace', 'reportedEthnicity', 'inferred_ethnicity', 'centerID', 'CEBPA_Biallelic', 'consensusAMLFusions', 'ageAtDiagnosis', 'isRelapse', 'isDenovo', 'isTransformed', 'specificDxAtAcquisition_MDSMPN', 'nonAML_MDSMPN_specificDxAtAcquisition', 'priorMalignancyNonMyeloid', 'priorMalignancyType', 'cumulativeChemo', 'priorMalignancyRadiationTx', 'priorMDS', 'priorMDSMoreThanTwoMths', 'priorMDSMPN', 'priorMDSMPNMoreThanTwoMths', 'priorMPN', 'priorMPNMoreThanTwoMths', 'dxAtInclusion', 'specificDxAtInclusion', 'ELN2017', 'dxAtSpecimenAcquisition', 'specificDxAtAcquisition', 'ageAtSpecimenAcquisition', 'timeOfSampleCollectionRelativeToInclusion', 'specimenGroups', 'diseaseStageAtSpecimenCollection', 'specimenType', 'rnaSeq', 'exomeSeq', 'totalDrug', 'analysisRnaSeq', 'an

In [53]:
# ── Build binary mutation matrix ───────────────────────────────────────────
AML_GENES = [
    # ── Signaling / kinase (proliferation) ───────────────────────────────
    "FLT3",    # most common — ITD + TKD
    "KIT",     # CBF-AML
    "NRAS",    # RAS pathway
    "KRAS",    # RAS pathway
    "PTPN11",  # RAS/MAPK, common in JMML/AML
    "CBL",     # ubiquitin ligase / RAS signaling

    # ── Transcription factors ─────────────────────────────────────────────
    "RUNX1",   # core binding factor
    "CEBPA",   # myeloid differentiation — biallelic = favorable
    "WT1",     # transcriptional repressor
    "ETV6",    # TEL — fusion partner + standalone mut

    # ── Epigenetic — DNA methylation ──────────────────────────────────────
    "DNMT3A",  # R882 hotspot
    "TET2",    # hydroxymethylation
    "IDH1",    # R132 hotspot — neomorphic
    "IDH2",    # R140/R172 hotspots — neomorphic

    # ── Epigenetic — histone modification ─────────────────────────────────
    "ASXL1",   # PRC2 complex — adverse prognosis
    "EZH2",    # H3K27 methyltransferase
    "KDM6A",   # H3K27 demethylase
    "SETD2",   # H3K36 methyltransferase
    "KMT2A",   # MLL — rearrangement + PTD

    # ── Chromatin remodeling ──────────────────────────────────────────────
    "STAG2",   # cohesin
    "RAD21",   # cohesin
    "SMC1A",   # cohesin
    "SMC3",    # cohesin

    # ── Nucleophosmin / NPM ───────────────────────────────────────────────
    "NPM1",    # most common in normal karyotype AML

    # ── Tumor suppressor ──────────────────────────────────────────────────
    "TP53",    # complex karyotype — very adverse
    "WT1",     # already listed above

    # ── Splicing ──────────────────────────────────────────────────────────
    "SRSF2",   # secondary AML / MDS-AML
    "SF3B1",   # MDS-AML
    "U2AF1",   # splicing factor
    "ZRSR2",   # splicing factor — X-linked

    # ── Cohesion / genome stability ───────────────────────────────────────
    "BCOR",    # transcriptional corepressor
    "BCORL1",

    # ── Other recurrent ───────────────────────────────────────────────────
    "PHF6",    # X-linked — T-AML / mixed
    "GATA2",   # germline predisposition + somatic
]

# Remove duplicates (WT1 listed twice above as example)
AML_GENES = list(dict.fromkeys(AML_GENES))


In [54]:
mut_raw = pd.read_csv(mut_path, sep="\t")
print(f"Mutation file shape    : {mut_raw.shape}")
print(f"dbgap_sample_id sample : {mut_raw['dbgap_sample_id'].iloc[:3].tolist()}")

# Map BA*D -> BA*R
mut_raw['rnaseq_id'] = mut_raw['dbgap_sample_id'].map(wes_to_rna)
n_unmapped = mut_raw['rnaseq_id'].isna().sum()
print(f"Rows unmapped to RNA   : {n_unmapped} / {len(mut_raw)} (expected for WES-only samples)")
mut_raw = mut_raw.dropna(subset=['rnaseq_id'])


Mutation file shape    : (11721, 32)
dbgap_sample_id sample : ['BA2336D', 'BA2336D', 'BA2336D']
Rows unmapped to RNA   : 3636 / 11721 (expected for WES-only samples)


In [55]:
mut_matrix = (
    mut_raw[['rnaseq_id', 'symbol']]
    .drop_duplicates()
    .assign(mutated=1)
    .pivot_table(index='rnaseq_id', columns='symbol', values='mutated', fill_value=0)
)
mut_matrix.columns.name = None
mut_matrix.index.name   = None

genes_present = [g for g in AML_GENES if g in mut_matrix.columns]
genes_missing = [g for g in AML_GENES if g not in mut_matrix.columns]
mutation_labels = mut_matrix[genes_present]

print(f"Mutation matrix : {mutation_labels.shape}")
print(f"Genes found     : {genes_present}")
if genes_missing:
    print(f"Genes missing   : {genes_missing}")


Mutation matrix : (637, 33)
Genes found     : ['FLT3', 'KIT', 'NRAS', 'KRAS', 'PTPN11', 'CBL', 'RUNX1', 'CEBPA', 'WT1', 'ETV6', 'DNMT3A', 'TET2', 'IDH1', 'IDH2', 'ASXL1', 'EZH2', 'KDM6A', 'SETD2', 'KMT2A', 'STAG2', 'RAD21', 'SMC1A', 'SMC3', 'NPM1', 'TP53', 'SRSF2', 'SF3B1', 'U2AF1', 'ZRSR2', 'BCOR', 'BCORL1', 'PHF6', 'GATA2']


In [56]:
# ── Identify confirmed normals present in bulk ─────────────────────────────
normals_in_bulk = sorted(
    s for s in normal_rna_ids if s in adata_bulk.obs_names
)
print(f"Confirmed normal samples in bulk : {len(normals_in_bulk)}")

# ── Build all-zero label rows for confirmed normals ────────────────────────
zero_labels = pd.DataFrame(
    0,
    index=normals_in_bulk,
    columns=mutation_labels.columns,
    dtype=np.float32,
)

# ── Merge WES-labeled AML samples + confirmed normals ─────────────────────
# If a normal happens to also appear in WES overlap, keep the WES labels.
mutation_labels_full = pd.concat([mutation_labels, zero_labels])
mutation_labels_full = mutation_labels_full[
    ~mutation_labels_full.index.duplicated(keep='first')
]

# ── Exclude AML samples with no WES calls and not confirmed normal ─────────
wes_samples    = set(mutation_labels.index)
train_samples  = set(mutation_labels_full.index)
all_bulk       = set(adata_bulk.obs_names)

excluded = sorted(all_bulk - train_samples)

print(f"\nSample breakdown:")
print(f"  WES-labeled AML samples     : {len(wes_samples & all_bulk)}")
print(f"  Confirmed normals added      : {len([s for s in normals_in_bulk if s not in wes_samples])}")
print(f"  Excluded (AML, no WES call)  : {len(excluded)}")
print(f"  Total training samples       : {len(train_samples & all_bulk)}")
if excluded:
    print(f"\n  Excluded sample IDs: {excluded[:10]}{'...' if len(excluded) > 10 else ''}")

# ── Subset adata_bulk and mutation_labels to training set ──────────────────
shared = adata_bulk.obs_names[adata_bulk.obs_names.isin(train_samples)]

assert len(shared) > 0, "No training samples found — check file paths and ID formats."

adata_bulk      = adata_bulk[shared].copy()
mutation_labels = mutation_labels_full.loc[shared]

# ── Annotate sample type in adata_bulk.obs ────────────────────────────────
adata_bulk.obs['sample_type'] = 'AML'
adata_bulk.obs.loc[
    adata_bulk.obs_names.isin(normals_in_bulk), 'sample_type'
] = 'Normal'

print(f"\nadata_bulk sample_type counts:")
print(adata_bulk.obs['sample_type'].value_counts())


Confirmed normal samples in bulk : 36

Sample breakdown:
  WES-labeled AML samples     : 615
  Confirmed normals added      : 36
  Excluded (AML, no WES call)  : 56
  Total training samples       : 651

  Excluded sample IDs: ['BA2022R', 'BA2050R', 'BA2094R', 'BA2095R', 'BA2096R', 'BA2134R', 'BA2168R', 'BA2172R', 'BA2175R', 'BA2228R']...

adata_bulk sample_type counts:
sample_type
AML       615
Normal     36
Name: count, dtype: int64


## 4. Load single-cell data & sanity checks

In [57]:
adata_sc = ad.read_h5ad(sc_path)
print(f"SC loaded : {adata_sc.n_obs} cells x {adata_sc.n_vars} genes")


SC loaded : 44823 cells x 27899 genes


In [58]:
# ── Sanity checks ──────────────────────────────────────────────────────────
X = adata_bulk.X
print(f"Bulk  : {adata_bulk.n_obs} samples x {adata_bulk.n_vars} genes")
print(f"SC    : {adata_sc.n_obs} cells x {adata_sc.n_vars} genes")
print(f"Muts  : {mutation_labels.shape}")
print()
print(f"NaN count  : {np.isnan(X).sum()}")
print(f"Inf count  : {np.isinf(X).sum()}")
print(f"Min value  : {np.nanmin(X):.3f}")
print(f"Max value  : {np.nanmax(X):.3f}")
print(f"Neg values : {(X < 0).sum()}  (expected if data is already log-normalised)")
print()
print("Mutation frequencies:")
print(mutation_labels.sum().sort_values(ascending=False))
mutation_labels.head()


Bulk  : 651 samples x 19975 genes
SC    : 44823 cells x 27899 genes
Muts  : (651, 33)

NaN count  : 0
Inf count  : 0
Min value  : -9.383
Max value  : 14.501
Neg values : 2745726  (expected if data is already log-normalised)

Mutation frequencies:
NPM1      141.0
DNMT3A    138.0
NRAS       96.0
TET2       82.0
RUNX1      75.0
IDH2       73.0
SRSF2      72.0
FLT3       67.0
ASXL1      63.0
TP53       57.0
IDH1       49.0
WT1        47.0
STAG2      43.0
CEBPA      37.0
SF3B1      35.0
BCOR       34.0
KRAS       34.0
PTPN11     32.0
U2AF1      32.0
GATA2      29.0
PHF6       21.0
EZH2       19.0
KIT        16.0
RAD21      15.0
CBL        13.0
BCORL1     13.0
SMC1A      10.0
SMC3       10.0
ZRSR2      10.0
ETV6        6.0
SETD2       4.0
KDM6A       4.0
KMT2A       2.0
dtype: float64


,FLT3,KIT,NRAS,KRAS,PTPN11,CBL,RUNX1,CEBPA,WT1,ETV6,...,NPM1,TP53,SRSF2,SF3B1,U2AF1,ZRSR2,BCOR,BCORL1,PHF6,GATA2
BA2392R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BA2611R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BA2506R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BA2430R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BA2448R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Phase 1 — Bulk pipeline

The dbGaP expression file is already log-normalised, so `norm_method='none'` and `log1p=False`.  
`n_components` is capped safely below `n_samples`.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Cap components at elbow heuristic — inspect scree first,
# then set manually. 50 is a reasonable default for ~450 samples.
#n_components = min(50, adata_bulk.n_obs - 1)
#n_components = min(120, adata_bulk.n_obs - 1)
n_components = min(160, adata_bulk.n_obs - 1)
print(f"Using n_components = {n_components}")

bulk_pipe = BulkPipeline(
    norm_method='none',
    log1p=False,
    center=True,
    scale=True,
    decomposition='svd',
    n_components=n_components,
    classifier='logistic',
    classifier_kwargs={
        'C': 1.0,                     # can try 0.1, 0.01, 1.0, 0.75
        'class_weight': 'balanced',
        'max_iter': 25000,
        'solver': 'saga',
        'random_state': 42,
        # l1_ratio removed — not supported with default l2 penalty
    },
)

bulk_pipe.fit(adata_bulk, mutation_labels, cv=5)


08:39:05 | INFO     | scope.pipeline.bulk_pipeline — === BulkPipeline.fit ===
08:39:05 | INFO     | scope.pipeline.bulk_pipeline — Preprocessing bulk data (norm=none, log1p=False).
08:39:05 | INFO     | scope.preprocessing.bulk — BulkNormalizer fitted (method=none).
08:39:05 | INFO     | scope.preprocessing.bulk — BulkScaler fitted (center=True, scale=True).


Using n_components = 160


08:39:05 | INFO     | scope.pipeline.bulk_pipeline — Decomposition: svd (k=160).
08:39:06 | INFO     | scope.decomposition.svd — SVD fitted: 160 components (cumulative EVR=0.849).
08:39:06 | INFO     | scope.pipeline.bulk_pipeline — Training classifiers (logistic).
/opt/homebrew/Cellar/micromamba/2.5.0_1/envs/scope-dev/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
08:39:06 | INFO     | scope.classification.base — Trained classifier for 'FLT3' (n_pos=67 / n_tot=651).
08:39:06 | WARNING  | scope.classification.base — Skipping 'KIT': positive fraction=0.025 (threshold=0.050).
/opt/homebrew/Cellar/micromamba/2.5.0_1/envs/scope-dev/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
08:39:07 | INFO     | scope.classification.base — Train

In [ ]:
# ── CV results summary ─────────────────────────────────────────────────────
summary = (
    bulk_pipe.cv_results_
    .groupby('mutation')[['auroc', 'auprc', 'brier']]
    .agg(['mean', 'std'])
    .round(3)
)

print(summary)


In [ ]:
import matplotlib as mpl

mpl.rcParams['figure.dpi'] = 100        # inline display quality
mpl.rcParams['savefig.dpi'] = 300       # saved file quality
mpl.rcParams['figure.figsize'] = (10, 6)  # optional: larger default size


In [ ]:
# ── Scree plot ─────────────────────────────────────────────────────────────
scree = bulk_pipe.decomposer_.scree_data()
fig, ax = plot_scree(scree, max_components=n_components)
plt.tight_layout()
plt.show()


## 6. Phase 2 — Single-cell projection

In [ ]:
# ── Gene space check ───────────────────────────────────────────────────────
print("Bulk var_names (first 10):", adata_bulk.var_names[:10].tolist())
print("SC   var_names (first 10):", adata_sc.var_names[:10].tolist())

overlap = adata_bulk.var_names.intersection(adata_sc.var_names)

print(f"\nShared genes (bulk intersect SC): {len(overlap)}")


In [ ]:
# ── Fit & transform single-cell pipeline ──────────────────────────────────
adata_bulk_pp = bulk_pipe.preprocessor_.transform(adata_bulk)

sc_pipe = SingleCellPipeline(
    bulk_pipeline=bulk_pipe,
    #alignment_method='z_score_bulk',
    alignment_method='moment_matching',
    #alignment_method='quantile',
    #alignment_method='none',
    sc_min_counts=100,
    sc_min_genes=100,
)

sc_pipe.fit(adata_bulk_pp, adata_sc)
adata_sc = sc_pipe.transform(adata_sc)

prob_cols = [c for c in adata_sc.obs.columns if c.startswith('mutation_prob_')]

print('Mutation probability columns:', prob_cols)
adata_sc.obs[prob_cols].describe()


## 7. UMAP visualisation

In [ ]:
adata_sc = compute_umap(adata_sc, obsm_key=bulk_pipe.obsm_key_, n_neighbors=15)


In [ ]:
fig = plot_mutation_probabilities(
    adata_sc,
    obsm_key='X_umap',
    ncols=3,
)

fig.savefig('mutation_probability_umap.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# Make more robust ranges for mutation prediction UMAPs
import scanpy as sc
import numpy as np

prob_cols = [c for c in adata_sc.obs.columns if c.startswith('mutation_prob_')]

# Per-gene vmax at 99th percentile so each panel uses its own dynamic range
vmaxes = {c: float(np.percentile(adata_sc.obs[c].values, 99)) for c in prob_cols}

# Floor vmaxes — if 99th pct is basically 0, use 0.1 so the colorbar isn't degenerate
vmaxes = {c: max(v, 0.0005) for c, v in vmaxes.items()}

print("Per-gene vmax (99th pct):")
for c, v in vmaxes.items():
    print(f"  {c.replace('mutation_prob_', ''):<8} {v:.3f}")

ncols = 3
nrows = int(np.ceil(len(prob_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten()

for i, col in enumerate(prob_cols):
    gene = col.replace('mutation_prob_', '')
    sc.pl.umap(
        adata_sc,
        color=col,
        ax=axes[i],
        show=False,
        title=f'P({gene} mut)',
        vmin=0,
        vmax=vmaxes[col],
        cmap='RdBu_r',
        colorbar_loc='right',
        s=2,  # point size — adjust if too small/large
    )

# Hide any unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig.savefig('mutation_probability_umap.pdf', bbox_inches='tight')
plt.show()


In [ ]:
import scanpy as sc

# Parse sample ID from barcode prefix
adata_sc.obs['sample'] = adata_sc.obs_names.str.split('_').str[0]

# Check what orig.ident looks like — may already be the sample label
#print(adata_sc.obs['sample'].value_counts())
#print(adata_sc.obs['orig.ident'].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 7))

# By sample
sc.pl.umap(adata_sc, color='sample', ax=axes[0], show=False, title='By sample')

# By cell type (useful reference alongside sample)
sc.pl.umap(adata_sc, color='CellType', ax=axes[1], show=False, title='By cell type')

plt.tight_layout()
plt.savefig('umap_by_sample.pdf', bbox_inches='tight')
plt.show()


## 8. Cluster-level summary (optional)

Run Leiden clustering via scanpy before this cell if desired.

In [ ]:
# import scanpy as sc
# sc.pp.neighbors(adata_sc, use_rep=bulk_pipe.obsm_key_)
# sc.tl.leiden(adata_sc)

if 'leiden' in adata_sc.obs.columns:
    fig, ax = plot_mutation_heatmap(adata_sc, cluster_key='leiden')
    plt.show()
else:
    print("No 'leiden' column found — run scanpy clustering above first.")
    

## 9. Other evaluation of results

In [ ]:
ncols = 5
nrows = int(np.ceil(len(prob_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.6, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(prob_cols):
    if col not in adata_sc.obs.columns:
        axes[i].set_visible(False)
        continue
    gene = col.replace('mutation_prob_', '')
    vals = adata_sc.obs[col].values
    axes[i].hist(vals, bins=50, color='steelblue', edgecolor='none')
    axes[i].axvline(np.percentile(vals, 95), color='red', linestyle='--', lw=1, label='95th pct')
    axes[i].set_title(gene)
    axes[i].set_xlabel('P(mut)')
    axes[i].set_ylabel('# cells')
    axes[i].legend(fontsize=7)

for j in range(len(prob_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Mutation probability distributions', y=1.02)
plt.tight_layout()
plt.savefig('prob_distributions.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── 2. Mean mutation probability per cell type ─────────────────────────────
celltype_means = (
    adata_sc.obs.groupby('CellType')[prob_cols]
    .mean()
    .rename(columns=lambda c: c.replace('mutation_prob_', ''))
    .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(celltype_means.T, aspect='auto', cmap='Reds')
ax.set_xticks(range(len(celltype_means)))
ax.set_xticklabels(celltype_means.index, rotation=45, ha='right')
ax.set_yticks(range(len(celltype_means.columns)))
ax.set_yticklabels(celltype_means.columns)
plt.colorbar(im, ax=ax, label='Mean P(mut)')
ax.set_title('Mean mutation probability by cell type')
plt.tight_layout()
plt.savefig('prob_by_celltype_heatmap.pdf', bbox_inches='tight')
plt.show()

print(celltype_means.round(4))


In [ ]:
# ── 3. Top cells by mutation probability ──────────────────────────────────
# For each gene, what cell types are enriched in the top 5% of cells?
print("Top 5% cell type enrichment per gene:\n")
for col in prob_cols:
    gene = col.replace('mutation_prob_', '')
    threshold = np.percentile(adata_sc.obs[col], 95)
    top_cells = adata_sc.obs[adata_sc.obs[col] >= threshold]
    ct_frac = top_cells['CellType'].value_counts(normalize=True).round(3)
    print(f"{gene} (threshold={threshold:.3f}, n={len(top_cells)}):")
    print(ct_frac.to_string())
    print()
    

In [ ]:
# ── 4. Mutation probability correlation matrix ─────────────────────────────
import seaborn as sns

corr = adata_sc.obs[prob_cols].corr()
corr.index   = corr.index.str.replace('mutation_prob_', '')
corr.columns = corr.columns.str.replace('mutation_prob_', '')

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Mutation probability correlation (per cell)')
plt.tight_layout()
plt.savefig('prob_correlation.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── 5. Per-sample mean probability (are some samples driving signal?) ──────
adata_sc.obs['sample'] = adata_sc.obs_names.str.split('_').str[0]

sample_means = (
    adata_sc.obs.groupby('sample')[prob_cols]
    .mean()
    .rename(columns=lambda c: c.replace('mutation_prob_', ''))
)

fig, ax = plt.subplots(figsize=(max(8, len(sample_means) * 0.8), 6))
im = ax.imshow(sample_means.T, aspect='auto', cmap='Reds')
ax.set_xticks(range(len(sample_means)))
ax.set_xticklabels(sample_means.index, rotation=45, ha='right')
ax.set_yticks(range(len(sample_means.columns)))
ax.set_yticklabels(sample_means.columns)
plt.colorbar(im, ax=ax, label='Mean P(mut)')
ax.set_title('Mean mutation probability by sample')
plt.tight_layout()
plt.savefig('prob_by_sample_heatmap.pdf', bbox_inches='tight')
plt.show()


## 10. Patient and mutation status-aware evaluations and visualizations

In [ ]:
# =============================================================================
# CHUNK 0 — Imports, constants, load metadata
# =============================================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scanpy as sc

#AML_GENES = ["FLT3", "NPM1", "DNMT3A", "IDH1", "IDH2", "TET2", "RUNX1", "TP53", "NRAS", "KRAS"]

# Always derive from what was actually fitted — source of truth
prob_cols = [c for c in adata_sc.obs.columns if c.startswith('mutation_prob_')]
AML_GENES = [c.replace('mutation_prob_', '') for c in prob_cols]

print(f"{len(AML_GENES)} genes with fitted classifiers:")
print(AML_GENES)

prob_cols  = [f'mutation_prob_{g}' for g in AML_GENES]

SUPP8_PATH = '/Users/ashforda/Desktop/Pathways + Omics/precepts/VanGalen_paper/supplemental_figures_and_tables/NIHMS1524068-supplement-8.xlsx'

GROUPS      = ['Normal BM', 'AML Diagnosis', 'AML Post-treatment', 'Cell line']
GROUP_COLORS = {
    'Normal BM':          '#4878CF',
    'AML Diagnosis':      '#D65F5F',
    'AML Post-treatment': '#F5A623',
    'Cell line':          '#9B59B6',
}

def normalise(s):
    return re.sub(r'[\s\-\._]', '', str(s)).upper()


In [ ]:
# =============================================================================
# CHUNK 1 — Parse Supplement 8
# =============================================================================

meta = pd.read_excel(SUPP8_PATH)

def parse_days(d):
    if pd.isna(d): return np.nan
    m = re.match(r'D(\d+)', str(d))
    return int(m.group(1)) if m else np.nan

meta['days_int'] = meta['Days from diagnosis'].apply(parse_days)

def assign_group(row):
    tissue = str(row.get('Tissue', '')).lower()
    sample = str(row.get('Sample', ''))
    days   = row.get('days_int', np.nan)
    if tissue == 'cell line':          return 'Cell line'
    if sample.startswith('BM'):        return 'Normal BM'
    if pd.isna(days) or days == 0:     return 'AML Diagnosis'
    return 'AML Post-treatment'

meta['group'] = meta.apply(assign_group, axis=1)

def parse_mutations(mut_str):
    if pd.isna(mut_str) or str(mut_str) in ('Not performed', 'None Detected', 'Unknown', 'nan'):
        return set()
    found = set()
    for gene in AML_GENES:
        if re.search(rf'\b{gene}[\s\-\.]', str(mut_str)) or str(mut_str).strip().startswith(gene):
            found.add(gene)
    return found

meta['known_mutations'] = meta['RHP Mutations'].apply(parse_mutations)
meta['sample_norm_key'] = meta['Sample'].apply(normalise)

print("Group distribution in metadata:")
print(meta['group'].value_counts())
print()
print(meta[['Sample', 'days_int', 'group', 'known_mutations']].to_string())


In [ ]:
# =============================================================================
# CHUNK 2 — Annotate adata_sc.obs
# =============================================================================

# ── Parse patient prefix and timepoint from barcode ───────────────────────
# SC format: 'AML314.D31_ACGTTTGG' → patient='AML314', timepoint='D31'
#            'BM5.34p_ACGT'        → patient='BM5.34p', timepoint=None
#            'OCI.AML3_ACGT'       → patient='OCI.AML3', timepoint=None

def parse_sample_info(obs_name):
    sample_field = obs_name.split('_')[0]
    m = re.match(r'^(.+?)\.(D\d+)$', sample_field)
    if m:
        prefix, timepoint = m.group(1), m.group(2)
    else:
        prefix, timepoint = sample_field, None
    return sample_field, prefix, timepoint, normalise(prefix)

parsed = list(map(parse_sample_info, adata_sc.obs_names))
adata_sc.obs['sample']      = [p[0] for p in parsed]  # e.g. 'AML314.D31'
adata_sc.obs['patient']     = [p[1] for p in parsed]  # e.g. 'AML314'
adata_sc.obs['timepoint']   = [p[2] for p in parsed]  # e.g. 'D31' or None
adata_sc.obs['patient_key'] = [p[3] for p in parsed]  # normalised for lookup

# ── Build group lookup from metadata using patient prefix ─────────────────
meta_sorted   = meta.sort_values('days_int', na_position='last')
norm_to_group = {}
for _, row in meta_sorted.iterrows():
    key = row['sample_norm_key']
    if key not in norm_to_group:
        norm_to_group[key] = row['group']

# Assign group — all AML cells start as Diagnosis, then refine by timepoint
adata_sc.obs['group'] = adata_sc.obs['patient_key'].map(norm_to_group)

# Cells with a non-D0 timepoint suffix → Post-treatment
post_tx_mask = (
    adata_sc.obs['timepoint'].notna() &
    (adata_sc.obs['timepoint'] != 'D0') &
    (adata_sc.obs['group'] == 'AML Diagnosis')
)
adata_sc.obs.loc[post_tx_mask, 'group'] = 'AML Post-treatment'

# Fallback for anything still unmatched
still_unmatched = adata_sc.obs['group'].isna()
if still_unmatched.sum():
    print("Still unmatched:", adata_sc.obs.loc[still_unmatched, 'patient'].unique())
    def infer_group(s):
        s = str(s).upper()
        if s.startswith('BM'):          return 'Normal BM'
        if s.startswith('AML'):         return 'AML Diagnosis'
        if 'OCI' in s or 'MUTZ' in s:  return 'Cell line'
        return 'Unknown'
    adata_sc.obs['group'] = adata_sc.obs['group'].fillna(
        adata_sc.obs['patient'].map(infer_group)
    )

# ── Build d0_mutation_map: patient_key → set of detected genes ───────────
# Uses D0 mutation calls; post-treatment cells inherit from their patient's D0
d0_mutation_map = {}
for _, row in meta[meta['group'].isin(['AML Diagnosis', 'Cell line'])].iterrows():
    key = row['sample_norm_key']
    d0_mutation_map.setdefault(key, set())
    d0_mutation_map[key] |= row['known_mutations']

adata_sc.obs['detected_mutations'] = adata_sc.obs['patient_key'].map(
    lambda k: d0_mutation_map.get(k, set())
)

# ── Verify ────────────────────────────────────────────────────────────────
print("Group counts:")
print(adata_sc.obs['group'].value_counts())
print("\nPer-sample group + mutations:")
print(adata_sc.obs.groupby('sample')[['group', 'detected_mutations']].first().to_string())


In [ ]:
# =============================================================================
# CHUNK 3 — Helper: get per-gene filtered cell mask
#
# Normal BM  → all cells always included
# AML Dx     → only cells from samples where gene was detected
# AML Post   → only cells from samples where gene was detected at D0
# Cell line  → only cells from cell lines where gene was detected
# =============================================================================

def get_gene_mask(adata, gene, group):
    """
    Returns boolean mask for cells to include for a given gene + group.
    """
    group_mask = adata.obs['group'] == group
    if group == 'Normal BM':
        return group_mask   # always include all normal cells
    # For other groups, additionally require mutation was detected in that sample
    mut_mask = adata.obs['detected_mutations'].apply(lambda s: gene in s)
    return group_mask & mut_mask

# Sanity check
print("\nPer-gene sample counts (mutation-positive only):\n")
rows = []
for gene in AML_GENES:
    row = {'gene': gene}
    for grp in GROUPS:
        mask = get_gene_mask(adata_sc, gene, grp)
        row[grp] = int(mask.sum())
    rows.append(row)
counts_df = pd.DataFrame(rows).set_index('gene')
print(counts_df.to_string())


In [ ]:
# =============================================================================
# CHUNK 4 — UMAP overview (all cells, colored by group + cell type)
# =============================================================================

plots = [
    ('sample',   'By sample',    {}),
    ('group',    'By group',     {'palette': GROUP_COLORS}),
    ('CellType', 'By cell type', {}),
]

ncols = 2
nrows = int(np.ceil(len(plots) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 10, nrows * 6))
axes = axes.flatten()

for i, (color, title, kwargs) in enumerate(plots):
    sc.pl.umap(adata_sc, color=color, ax=axes[i], show=False,
               title=title, legend_loc='right margin', s=2, **kwargs)

# Hide unused axes
for j in range(len(plots), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('umap_sample_overview.pdf', bbox_inches='tight')
plt.show()


In [ ]:
ncols = 5
nrows = int(np.ceil(len(AML_GENES) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4))
axes = axes.flatten()

for i, gene in enumerate(AML_GENES):
    ax  = axes[i]
    col = f'mutation_prob_{gene}'

    data_by_group = {}
    for grp in GROUPS:
        mask = get_gene_mask(adata_sc, gene, grp)
        vals = adata_sc.obs.loc[mask, col].values
        if len(vals) > 0:
            data_by_group[grp] = vals

    positions = list(range(len(data_by_group)))
    labels    = list(data_by_group.keys())
    vals_list = list(data_by_group.values())
    colors    = [GROUP_COLORS[g] for g in labels]

    parts = ax.violinplot(vals_list, positions=positions,
                          showmedians=True, showextrema=False)
    for pc, color in zip(parts['bodies'], colors):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(1.5)

    ax.set_xticks(positions)
    ax.set_xticklabels([l.replace(' ', '\n') for l in labels], fontsize=7)
    ax.set_title(gene, fontsize=10)
    ax.set_ylabel('P(mut)')

    for pos, (grp, vals) in enumerate(data_by_group.items()):
        ax.text(pos, ax.get_ylim()[0], f'n={len(vals)}',
                ha='center', va='bottom', fontsize=6, color='#555555')

for j in range(len(AML_GENES), len(axes)):
    axes[j].set_visible(False)

legend_handles = [mpatches.Patch(color=v, label=k) for k, v in GROUP_COLORS.items()]
axes[0].legend(handles=legend_handles, fontsize=6, loc='upper right')

plt.suptitle(
    'Mutation probabilities by group\n'
    '(AML/Cell line: mutation-positive samples only; Normal BM: all cells)',
    y=1.02, fontsize=11
)
plt.tight_layout()
plt.savefig('prob_violin_per_gene.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# =============================================================================
# CHUNK 6 — Per-gene distribution overlays (density histograms)
# =============================================================================

ncols = 5
nrows = int(np.ceil(len(AML_GENES) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4))
axes = axes.flatten()

for i, gene in enumerate(AML_GENES):
    col = f'mutation_prob_{gene}'
    for grp in GROUPS:
        mask = get_gene_mask(adata_sc, gene, grp)
        vals = adata_sc.obs.loc[mask, col].values
        if len(vals) == 0:
            continue
        axes[i].hist(vals, bins=40, alpha=0.45, density=True,
                     color=GROUP_COLORS[grp], label=f'{grp} (n={len(vals)})',
                     edgecolor='none')
    axes[i].set_title(gene, fontsize=10)
    axes[i].set_xlabel('P(mut)')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=5, loc='upper right')

for j in range(len(AML_GENES), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'Mutation probability distributions by group\n'
    '(AML/Cell line: mutation-positive samples only; Normal BM: all cells)',
    y=1.02, fontsize=11
)
plt.tight_layout()
plt.savefig('prob_dist_per_gene.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# =============================================================================
# CHUNK 7 — Per-gene UMAPs: one figure per gene, one panel per group
# =============================================================================

vmaxes = {c: max(float(np.percentile(adata_sc.obs[c].values, 99)), 0.1) for c in prob_cols}

for gene in AML_GENES:
    col = f'mutation_prob_{gene}'

    fig, axes = plt.subplots(1, len(GROUPS), figsize=(len(GROUPS) * 5, 5))

    for ax, grp in zip(axes, GROUPS):
        mask = get_gene_mask(adata_sc, gene, grp)
        sub  = adata_sc[mask].copy()
        n    = sub.n_obs

        if n == 0:
            ax.set_visible(False)
            continue

        sc.pl.umap(
            sub, color=col, ax=ax, show=False,
            title=f'{grp}\n(n={n})',
            vmin=0.0, vmax=vmaxes[col],
            cmap='Reds', colorbar_loc='right', s=12,
        )

    fig.suptitle(
        f'P({gene} mut) — mutation-positive samples per group',
        fontsize=13, y=1.02
    )
    plt.tight_layout()
    fname = f'umap_probs_{gene}.pdf'
    plt.savefig(fname, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")


In [ ]:
# =============================================================================
# CHUNK 8 — Summary heatmap: mean P(mut) per sample — raw + per-gene normed
# =============================================================================

def _draw_sample_heatmap(plot_data, row_colors, title, filename):
    """Draw a clustermap with fixed colorbar and group legend positioning."""
    figheight = max(8, len(plot_data) * 0.45)
    
    g = sns.clustermap(
        plot_data,
        row_colors=row_colors,
        col_cluster=False,
        row_cluster=True,
        cmap='Reds',
        figsize=(14, figheight),
        xticklabels=True,
        yticklabels=True,
        linewidths=0.3,
        dendrogram_ratio=0.1,
        cbar_pos=(1.02, 0.4, 0.02, 0.2),   # (left, bottom, width, height) — right side, middle
    )

    # Move colorbar label
    g.ax_cbar.set_ylabel('Mean P(mut)', fontsize=9)
    g.ax_cbar.tick_params(labelsize=8)

    # Group legend — anchored to heatmap axes, bottom right
    legend_handles = [mpatches.Patch(color=v, label=k) for k, v in GROUP_COLORS.items()]
    g.ax_heatmap.legend(
        handles=legend_handles,
        title='Group',
        loc='lower right',
        bbox_to_anchor=(1.0, -0.22),
        ncol=2,
        frameon=True,
        fontsize=8,
        title_fontsize=9,
    )

    g.fig.suptitle(title, y=1.01, fontsize=12)
    plt.savefig(filename, bbox_inches='tight')
    plt.show()
    print(f"Saved: {filename}")


# ── Build base sample means ────────────────────────────────────────────────
sample_means = (
    adata_sc.obs.groupby('sample')[prob_cols]
    .mean()
    .rename(columns=lambda c: c.replace('mutation_prob_', ''))
)

sample_to_group = adata_sc.obs.groupby('sample')['group'].first().to_dict()
sample_means['group'] = sample_means.index.map(sample_to_group).fillna('Unknown')
sample_means = sample_means.sort_values('group')

row_colors = sample_means['group'].map(GROUP_COLORS).fillna('#AAAAAA')
plot_data  = sample_means.drop(columns='group')


# ── Version 1: raw probabilities ──────────────────────────────────────────
_draw_sample_heatmap(
    plot_data,
    row_colors,
    title='Mean mutation probability per sample',
    filename='prob_heatmap_per_sample.pdf',
)


# ── Version 2: per-gene min-max normalised (0–1 within each gene column) ──
col_min  = plot_data.min()
col_max  = plot_data.max()
col_range = col_max - col_min
col_range[col_range == 0] = 1   # avoid divide-by-zero for flat columns

plot_data_normed = (plot_data - col_min) / col_range

_draw_sample_heatmap(
    plot_data_normed,
    row_colors,
    title='Mean mutation probability per sample (per-gene normalised 0–1)',
    filename='prob_heatmap_per_sample_normed.pdf',
)


## 11. Validate predictions to known van Galen single-cell mutation calls

In [ ]:
# =============================================================================
# CHUNK V0 — Inspect van Galen mutation call format
# =============================================================================

# Check PredictionRefined too — may be useful
print("PredictionRefined unique values:")
print(adata_sc.obs['PredictionRefined'].value_counts(dropna=False).head(20).to_string())

print("\nMutTranscripts — sample non-NA values:")
mut_nonempty = adata_sc.obs['MutTranscripts'][
    (adata_sc.obs['MutTranscripts'] != 'NA') &
    (adata_sc.obs['MutTranscripts'] != '')
]
print(mut_nonempty.value_counts().head(20).to_string())

print("\nWtTranscripts — sample non-NA values:")
wt_nonempty = adata_sc.obs['WtTranscripts'][
    (adata_sc.obs['WtTranscripts'] != 'NA') &
    (adata_sc.obs['WtTranscripts'] != '')
]
print(wt_nonempty.value_counts().head(20).to_string())

# Extract all unique gene names from both columns
import re

def extract_gene(val):
    if pd.isna(val) or val in ('NA', ''):
        return None
    m = re.match(r'^([A-Z0-9]+)\.', str(val))
    return m.group(1) if m else None

mut_genes = set(filter(None, adata_sc.obs['MutTranscripts'].map(extract_gene)))
wt_genes  = set(filter(None, adata_sc.obs['WtTranscripts'].map(extract_gene)))

print(f"\nGenes with mut transcript calls : {sorted(mut_genes)}")
print(f"Genes with WT transcript calls  : {sorted(wt_genes)}")
print(f"\nOverlap with AML_GENES:")
print(f"  Mut: {sorted(mut_genes & set(AML_GENES))}")
print(f"  WT:  {sorted(wt_genes  & set(AML_GENES))}")


In [ ]:
# =============================================================================
# CHUNK V1 — Parse MutTranscripts / WtTranscripts into per-gene status
#
# Logic per cell per gene:
#   MutTranscripts contains GENE.* → 'Mut'
#   WtTranscripts  contains GENE.* → 'WT'
#   both present                   → 'Mut' (mutant allele detected = call Mut)
#   neither                        → 'NA'
# =============================================================================

def extract_gene(val):
    if pd.isna(val) or str(val) in ('NA', ''):
        return None
    m = re.match(r'^([A-Z0-9]+)\.', str(val))
    return m.group(1) if m else None

# Build per-cell gene arrays once — avoids re-parsing in a loop
mut_gene_per_cell = adata_sc.obs['MutTranscripts'].map(extract_gene).values
wt_gene_per_cell  = adata_sc.obs['WtTranscripts'].map(extract_gene).values

STATUS_PALETTE = {'Mut': '#D62728', 'WT': '#1F77B4', 'NA': '#CCCCCC'}
STATUS_ORDER   = ['Mut', 'WT', 'NA']

validated_genes = []

for gene in AML_GENES:
    if gene not in (mut_genes | wt_genes):
        continue  # no SC calls for this gene

    status = np.where(
        mut_gene_per_cell == gene, 'Mut',
        np.where(wt_gene_per_cell == gene, 'WT', 'NA')
    )
    col = f'sc_status_{gene}'
    adata_sc.obs[col] = pd.Categorical(status, categories=STATUS_ORDER)

    counts = pd.Series(status).value_counts()
    print(f"{gene:10s}  Mut={counts.get('Mut', 0):5d}  "
          f"WT={counts.get('WT', 0):5d}  NA={counts.get('NA', 0):5d}")
    validated_genes.append(gene)

print(f"\nValidated genes ({len(validated_genes)}): {validated_genes}")


In [ ]:
# =============================================================================
# CHUNK V2 — Paired UMAPs: known SC status (left) vs predicted P(mut) (right)
# =============================================================================

from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

if not validated_genes:
    print("No validated genes — check CHUNK V1.")
else:
    umap_coords = adata_sc.obsm['X_umap']
    pt_size     = 1.5

    for gene in validated_genes:
        status_col = f'sc_status_{gene}'
        prob_col   = f'mutation_prob_{gene}'
        if prob_col not in adata_sc.obs.columns:
            print(f"No probability column for {gene}, skipping.")
            continue

        fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13, 5))

        status_vals = adata_sc.obs[status_col].values

        # ── Left: known status ────────────────────────────────────────────
        for status in ['NA', 'WT', 'Mut']:
            mask  = status_vals == status
            alpha = 0.25 if status == 'NA' else 0.85
            size  = pt_size * 0.5 if status == 'NA' else pt_size
            ax_l.scatter(
                umap_coords[mask, 0], umap_coords[mask, 1],
                c=STATUS_PALETTE[status], s=size,
                alpha=alpha, rasterized=True,
            )

        legend_elements = [
            Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=STATUS_PALETTE[s],
                   markersize=8, label=s)
            for s in STATUS_ORDER
        ]
        ax_l.legend(handles=legend_elements, fontsize=9, loc='lower right')
        ax_l.set_title(f'{gene} — known SC status', fontsize=11)
        ax_l.axis('off')

        # ── Right: predicted probability ──────────────────────────────────
        probs = adata_sc.obs[prob_col].values
        vmax  = max(float(np.percentile(probs, 99)), 1e-4)
        sc_   = ax_r.scatter(
            umap_coords[:, 0], umap_coords[:, 1],
            c=probs, cmap='Reds',
            vmin=0, vmax=vmax,
            s=pt_size, alpha=0.8, rasterized=True,
        )
        plt.colorbar(sc_, ax=ax_r, label='P(mut)', shrink=0.75)
        ax_r.set_title(f'{gene} — predicted P(mut)', fontsize=11)
        ax_r.axis('off')

        fig.suptitle(f'Validation: {gene}', fontsize=13, y=1.02)
        plt.tight_layout()
        fname = f'validation_umap_{gene}.pdf'
        plt.savefig(fname, bbox_inches='tight', dpi=150)
        plt.show()
        print(f"Saved: {fname}")
        

In [ ]:
# =============================================================================
# CHUNK V3 — Violin plots: WT vs Mut predicted probability + MWU p-values
# =============================================================================

from scipy import stats

def mwu_pval(a, b):
    if len(a) < 3 or len(b) < 3:
        return np.nan
    _, p = stats.mannwhitneyu(a, b, alternative='less')  # WT < Mut expected
    return p

def pval_label(p):
    if np.isnan(p):   return 'n<3'
    if p < 0.0001:    return '****'
    if p < 0.001:     return '***'
    if p < 0.01:      return '**'
    if p < 0.05:      return '*'
    return f'ns\n(p={p:.3f})'

if not validated_genes:
    print("No validated genes — check CHUNK V1.")
else:
    ncols  = min(3, len(validated_genes))
    nrows  = int(np.ceil(len(validated_genes) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 4.5, nrows * 5))
    axes = np.array(axes).flatten()

    summary_rows = []

    for i, gene in enumerate(validated_genes):
        ax         = axes[i]
        status_col = f'sc_status_{gene}'
        prob_col   = f'mutation_prob_{gene}'

        wt_vals  = adata_sc.obs.loc[adata_sc.obs[status_col] == 'WT',  prob_col].values
        mut_vals = adata_sc.obs.loc[adata_sc.obs[status_col] == 'Mut', prob_col].values
        na_vals  = adata_sc.obs.loc[adata_sc.obs[status_col] == 'NA',  prob_col].values

        groups     = [('WT',  wt_vals,  STATUS_PALETTE['WT']),
                      ('Mut', mut_vals, STATUS_PALETTE['Mut']),
                      ('NA',  na_vals,  STATUS_PALETTE['NA'])]
        groups     = [(l, v, c) for l, v, c in groups if len(v) > 0]

        labels_    = [f'{l}\n(n={len(v)})' for l, v, c in groups]
        vals_list  = [v for _, v, _ in groups]
        colors_    = [c for _, _, c in groups]
        positions  = list(range(len(groups)))

        parts = ax.violinplot(vals_list, positions=positions,
                              showmedians=True, showextrema=False)
        for pc, col in zip(parts['bodies'], colors_):
            pc.set_facecolor(col)
            pc.set_alpha(0.7)
        parts['cmedians'].set_color('black')
        parts['cmedians'].set_linewidth(2)

        # Jittered points (subsampled)
        rng = np.random.default_rng(42)
        for pos, (vals, col) in enumerate(zip(vals_list, colors_)):
            sub = vals if len(vals) <= 300 else rng.choice(vals, 300, replace=False)
            jitter = rng.uniform(-0.08, 0.08, len(sub))
            ax.scatter(pos + jitter, sub, s=4, alpha=0.35,
                       color=col, rasterized=True, zorder=3)

        ax.set_xticks(positions)
        ax.set_xticklabels(labels_, fontsize=8)
        ax.set_ylabel('P(mut)', fontsize=9)
        ax.set_title(gene, fontsize=11, fontweight='bold')

        # ── Significance bar: WT (pos 0) vs Mut (pos 1) ──────────────────
        p = mwu_pval(wt_vals, mut_vals)
        label_p = pval_label(p)

        if len(wt_vals) >= 3 and len(mut_vals) >= 3:
            all_vals = np.concatenate(vals_list)
            y_max  = np.percentile(all_vals, 99.5)
            y_bar  = y_max * 1.08
            y_text = y_max * 1.12
            ax.plot([0, 1], [y_bar, y_bar], color='black', lw=1.2)
            ax.plot([0, 0], [y_bar * 0.99, y_bar], color='black', lw=1.2)
            ax.plot([1, 1], [y_bar * 0.99, y_bar], color='black', lw=1.2)
            ax.text(0.5, y_text, label_p, ha='center', va='bottom',
                    fontsize=10, fontweight='bold')

        summary_rows.append({
            'gene':       gene,
            'n_WT':       len(wt_vals),
            'n_Mut':      len(mut_vals),
            'n_NA':       len(na_vals),
            'median_WT':  np.median(wt_vals)  if len(wt_vals)  > 0 else np.nan,
            'median_Mut': np.median(mut_vals) if len(mut_vals) > 0 else np.nan,
            'MWU_p':      p,
            'sig':        label_p,
        })

    for j in range(len(validated_genes), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        'Predicted P(mut): WT vs Mut cells (Mann-Whitney U, one-sided: WT < Mut)\n'
        '**** p<0.0001   *** p<0.001   ** p<0.01   * p<0.05',
        fontsize=10, y=1.02
    )
    plt.tight_layout()
    plt.savefig('validation_violin_wt_vs_mut.pdf', bbox_inches='tight', dpi=150)
    plt.show()

    # ── Summary table ─────────────────────────────────────────────────────
    summary_df = pd.DataFrame(summary_rows).set_index('gene')
    print("\nValidation summary:")
    print(summary_df.round(6).to_string())
    summary_df.to_csv('validation_summary.csv')
    print("Saved: validation_summary.csv")
    

## 12. Save outputs

In [ ]:
os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)

bulk_pipe.save('models/bulk_pipeline.pkl')

# Convert set column to comma-separated string for h5ad compatibility
adata_sc.obs['detected_mutations'] = adata_sc.obs['detected_mutations'].apply(
    lambda s: ','.join(sorted(s)) if isinstance(s, set) else ''
)

adata_sc.write_h5ad('results/sc_with_mutation_probs.h5ad')
print("Saved: models/bulk_pipeline.pkl")
print("Saved: results/sc_with_mutation_probs.h5ad")
